<h1>LD Pipeline 2024 (Version 1.0)</h1>

<h3>Python-Bibliotheken laden</h3>

In [1]:
import os
import sys

sys.path.append(os.path.abspath(".."))
from pathlib import Path
from pipeline.base import Env, Environment

config_path = Path.cwd().parent / "config.ini"
_e = Env.int
env = Environment(_e, config_path)

In [2]:
import os
import sys
import logging

sys.path.append(os.path.abspath(".."))
from pipeline.base import Utils

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    stream=sys.stdout,
    force=True,
)

try:
    initialized
except NameError:
    initialized = False

    # if not initialized:
    initialized = True
    os.chdir("../")

logging.info(f"hallo {env.name}!")

2025-10-16 16:03:06,003 - root - INFO - hallo int!


<h3><span style="
    display: inline-block;
    width: 50px;
    height: 50px;
    border-radius: 50%;
    border: 3px solid black;
    font-size: 24px;
    font-weight: bold;
    text-align: center;
    line-height: 50px;
    background-color: yellow;
">
    1
</span> Überprüfe des Start-Signals</h3>
<p style="max-width: 700px;">
    Es wird überprüft, ob im Ordner <b>\\\szh.loc\ssz\applikationen\HDB_Dropzone\PROD\Test\Pipeline</b> ein Start-Signal vorhanden ist. Falls ja, wird das Start-Signal in ein Running-Signal umgewandelt und die Pipeline beginnt mit der Ausführung.
</p>

In [3]:
import main

utils = Utils()

print(env.name)
print(os.getcwd())
print(env.config.get("start_signal_folder"))
utils.logger.info(f"Pipeline initialized for {env.name}")

if utils.check_start_signal(env):
    print("Start signal found and renamed to a running signal.")
else:
    print("No start signal found.")

int
g:\sszrul\ld-pipeline-2024
//szh.loc/ssz/applikationen/HDB_Dropzone/INT/Test/Pipeline
2025-10-16 16:03:09,940 - Utils - INFO - Pipeline initialized for int
No start signal found.


<h3><span style="
    display: inline-block;
    width: 50px;
    height: 50px;
    border-radius: 50%;
    border: 3px solid black;
    font-size: 24px;
    font-weight: bold;
    text-align: center;
    line-height: 50px;
    background-color: yellow;
">
    2
</span> Aktualisiere die Pipe-Tabellen</h3>
<p>
    Die relevanten Daten werden aus der HDB in Tabellen mit dem Prefix "pipe_" kopiert, um sie in einen konsistenten Zustand zu archivieren.
</p>

In [4]:
main.step(name="initPipeTables", env=env)

2025-10-16 16:03:09,994 - main - INFO - Running step initPipeTables
2025-10-16 16:03:09,997 - Pipeline - INFO - Initialized pipeline for 'int'
2025-10-16 16:03:09,999 - InitPipeTables - INFO - Initializing pipe tables for INT ...
2025-10-16 16:03:10,090 - Environment - INFO - Initializing db connection for mssql
2025-10-16 16:03:10,505 - MSSQLDbConnection - INFO - Database connection to szhm32652\SSZ_HDB:1433/SSZ_HDB established...
2025-10-16 16:03:10,622 - InitPipeTables - INFO - Executing pipe_HDBCodeliste.sql for pipe_pipe_HDBCodeliste ...
2025-10-16 16:03:11,163 - InitPipeTables - INFO - Done
2025-10-16 16:03:11,286 - InitPipeTables - INFO - Executing pipe_HDBCubeDefinition.sql for pipe_pipe_HDBCubeDefinition ...
2025-10-16 16:03:11,319 - InitPipeTables - INFO - Done
2025-10-16 16:03:11,430 - InitPipeTables - INFO - Executing pipe_HDBGruppenliste.sql for pipe_pipe_HDBGruppenliste ...
2025-10-16 16:03:11,635 - InitPipeTables - INFO - Done
2025-10-16 16:03:11,758 - InitPipeTables - I

In [5]:
main.step(name="copyHDBToPipeTables", env=env)

2025-10-16 16:03:20,913 - main - INFO - Running step copyHDBToPipeTables
2025-10-16 16:03:20,917 - Pipeline - INFO - Initialized pipeline for 'int'
2025-10-16 16:03:20,918 - CopyHDBToPipeTables - INFO - Copying HDB data to the "pipe" tables ...
2025-10-16 16:03:20,920 - Environment - INFO - Initializing db connection for mssql
2025-10-16 16:03:21,042 - MSSQLDbConnection - INFO - Database connection to szhm32652\SSZ_HDB:1433/SSZ_HDB established...
2025-10-16 16:03:21,043 - CopyHDBToPipeTables - INFO - Copying HDBCodeliste to pipe_HDBCodeliste ...

                                INSERT INTO pipe_HDBCodeliste
                                SELECT "CODE", "CODENAME", "REFERENZTABELLE", "Index" FROM #pipe_HDBCodeliste
                            
2025-10-16 16:03:23,504 - CopyHDBToPipeTables - INFO - Done
2025-10-16 16:03:23,505 - CopyHDBToPipeTables - INFO - Copying HDBCubeDefinition to pipe_HDBCubeDefinition ...

                                INSERT INTO pipe_HDBCubeDefinition
       

<h3><span style="
    display: inline-block;
    width: 50px;
    height: 50px;
    border-radius: 50%;
    border: 3px solid black;
    font-size: 24px;
    font-weight: bold;
    text-align: center;
    line-height: 50px;
    background-color: yellow;
">
    3
</span> Generiere die Triple-Dateien</h3>

In [6]:
triple_types_metadata = [
    "code",
    "cube",
    "groupCode",
    "hierarchy",
    "legalFoundation",
    "measureUnit",
    "measure",
    "property",
    "room",
    "time",
]
triple_types_observations = ["observation"]
triple_types_others = ["copyStatic", "buildTermsetHierarchy", "generateViews"]

# just for testing
triple_types_metadata = ["groupCode"]
triple_types_observations = []
triple_types_others = []


options_batching = {
    "db_batch_size": 100000,
    "write_batch_size": 600000,
    "max_iteration": None,
}

for name in triple_types_metadata:
    main.step(name=f"{name}Templating", env=env, options=options_batching)
for name in triple_types_observations:
    main.step(name=f"{name}Templating", env=env, options=options_batching)
for name in triple_types_others:
    main.step(name=name, env=env)

2025-10-16 16:03:37,959 - main - INFO - Running step groupCodeTemplating
2025-10-16 16:03:37,961 - Pipeline - INFO - Initialized pipeline for 'int'
2025-10-16 16:03:37,963 - Environment - INFO - Initializing db connection for mssql
2025-10-16 16:03:38,099 - MSSQLDbConnection - INFO - Database connection to szhm32652\SSZ_HDB:1433/SSZ_HDB established...
2025-10-16 16:03:38,250 - TemplatingOptimized - INFO - Creating temporary table #view_group_code_int ...
2025-10-16 16:03:39,706 - TemplatingOptimized - INFO - Creating an index on the _sort_order column ...
2025-10-16 16:03:39,833 - TemplatingOptimized - INFO - done
2025-10-16 16:03:40,514 - TemplatingOptimized - INFO - Downloading data ...
2025-10-16 16:03:42,536 - TemplatingOptimized - INFO - 100000 rows downloaded in the 1. iteration.
2025-10-16 16:03:42,536 - TemplatingOptimized - INFO - Generating triples ...
2025-10-16 16:03:47,502 - TemplatingOptimized - INFO - done
2025-10-16 16:03:47,503 - TemplatingOptimized - INFO - 1. iterati

<h3><span style="
    display: inline-block;
    width: 50px;
    height: 50px;
    border-radius: 50%;
    border: 3px solid black;
    font-size: 24px;
    font-weight: bold;
    text-align: center;
    line-height: 50px;
    background-color: yellow;
">
    4
</span> Setze das Start-Signal für die Generierung der Fuseki-Indexdateien</h3>

In [7]:
utils = Utils()
utils.set_start_signal_fuseki_index(env)

<h3><span style="
    display: inline-block;
    width: 50px;
    height: 50px;
    border-radius: 50%;
    border: 3px solid black;
    font-size: 24px;
    font-weight: bold;
    text-align: center;
    line-height: 50px;
    background-color: yellow;
">
    5
</span> Schreibe die Publikationsstati zurück in die HDB</h3>

In [8]:
main.step(name="writePublicationStatiToHDB", env=env)

2025-10-16 16:03:55,111 - main - INFO - Running step writePublicationStatiToHDB
2025-10-16 16:03:55,115 - Pipeline - INFO - Initialized pipeline for 'int'
2025-10-16 16:03:55,119 - WritePublicationStatiToHDB - INFO - Calculating observation hashes ...
2025-10-16 16:03:55,121 - Environment - INFO - Initializing db connection for mssql
2025-10-16 16:03:55,267 - MSSQLDbConnection - INFO - Database connection to szhm32652\SSZ_HDB:1433/SSZ_HDB established...
2025-10-16 16:04:44,542 - MSSQLDbConnection - INFO - Database connection to szhm32652\SSZ_HDB:1433/SSZ_HDB closed
2025-10-16 16:04:44,543 - WritePublicationStatiToHDB - INFO - Done
2025-10-16 16:04:44,544 - Environment - INFO - Initializing db connection for mssql
2025-10-16 16:04:44,719 - MSSQLDbConnection - INFO - Database connection to szhm32652\SSZ_HDB:1433/SSZ_HDB established...
2025-10-16 16:04:44,731 - WritePublicationStatiToHDB - INFO - Creating temporary table #hash_HDB_TEST ...
2025-10-16 16:04:53,198 - WritePublicationStatiTo

<h3><span style="
    display: inline-block;
    width: 50px;
    height: 50px;
    border-radius: 50%;
    border: 3px solid black;
    font-size: 24px;
    font-weight: bold;
    text-align: center;
    line-height: 50px;
    background-color: yellow;
">
    6
</span> Setze das Running-Signal auf Finish</h3>

In [9]:
utils = Utils()
utils.set_finish_signal(env)

False